# Parquet — a deep dive

_weyland notebook library · file-format series (01 of the Parquet / Arrow / Avro / Lance set)_

This notebook is **self-contained**: it builds a few hundred rows of sample data in
memory, writes them to Parquet on local disk, and then takes the file apart to show
*why* the format behaves the way it does. No lakeFS, no S3, no network — it runs
anywhere `polars`, `pyarrow`, and `duckdb` are installed.

**What you'll see**

1. What Parquet is — columnar vs row storage, and when to reach for it
2. Writing sample data to Parquet (pyarrow **and** polars) and finding it on disk
3. **Internals** — row groups, column chunks, schema, encodings, compression
4. A compression bake-off (uncompressed / snappy / zstd / gzip) on the same data
5. **Why columnar wins** — column projection and predicate (row-group) pushdown, measured
6. Round-trip read-back and equality check
7. When to use Parquet vs Arrow / Avro / Lance — the trade-offs

> Teaching notebook — it doubles as living documentation, so the prose matters as
> much as the code. Read top-to-bottom.


## 1 · What Parquet is

**Apache Parquet** is an open, language-neutral, **columnar** on-disk file format for
analytical (OLAP) data. It began as a Hadoop-ecosystem format and is now the
lingua franca of the data-lake world — Spark, Trino, DuckDB, polars, pandas,
BigQuery, Snowflake, and every Iceberg/Delta/Hudi table underneath all speak it.

### Row storage vs columnar storage

Imagine a table with columns `id, name, country, amount`. Two ways to lay those
bytes on disk:

```
Row-oriented (CSV, Avro, a classic OLTP heap):
  [1,"ada","GB",10.0] [2,"linus","FI",20.0] [3,"guido","NL",30.0] ...
  → all fields of row 1, then all fields of row 2, ...

Column-oriented (Parquet, Arrow, ORC):
  ids:       [1, 2, 3, ...]
  names:     ["ada","linus","guido", ...]
  countries: ["GB","FI","NL", ...]
  amounts:   [10.0, 20.0, 30.0, ...]
  → all of column A, then all of column B, ...
```

That single layout choice buys two things that dominate analytics performance:

- **Column projection** — a query that touches 2 of 12 columns reads only those 2
  columns off disk. In a row format you must read (and skip past) every field of
  every row.
- **Better compression** — values in one column share a type and usually a
  distribution (all countries, all prices). Homogeneous runs compress far harder
  than a heterogeneous row does, and enable per-column encodings like dictionary
  and run-length encoding.

Parquet adds a third: **per-column statistics** (min/max/null-count) stored per
*row group*, which lets a reader **skip whole blocks** that can't match a filter —
"predicate pushdown."

### When to reach for Parquet

**Good fit**

- Analytical scans over large tables — aggregations, `GROUP BY`, column subsets
- The storage format for a data lake / lakehouse (Iceberg, Delta, Hudi are Parquet underneath)
- Long-lived, write-once/read-many datasets where read speed and size matter
- Interchange between engines — everything reads Parquet

**Poor fit**

- Row-at-a-time OLTP: point lookups, single-row inserts/updates (use a database)
- Streaming append of one record at a time (use Avro or a log; batch, then write Parquet)
- Tiny data where the metadata footer overhead isn't worth it
- In-memory zero-copy compute hand-off (that's **Arrow** — the in-memory sibling)


## 2 · Build sample data and write it to Parquet

We fabricate ~500 rows of a plausible "orders" table entirely in memory with numpy +
polars. Deterministic seed so the notebook is reproducible. Note the deliberately
**low-cardinality** columns (`country`, `category`, `status`) — those are what
dictionary + RLE encoding will crush later.

In [1]:
import os, io, time, glob, tempfile
import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

print("polars ", pl.__version__)
print("pyarrow", pa.__version__)

rng = np.random.default_rng(42)
N = 500

countries  = np.array(["GB", "FI", "NL", "US", "DE", "JP"])
categories = np.array(["books", "electronics", "grocery", "toys"])
statuses   = np.array(["placed", "shipped", "delivered", "cancelled"])

df = pl.DataFrame({
    "order_id":  np.arange(1, N + 1, dtype=np.int64),
    "customer":  rng.integers(1000, 1050, size=N),          # ~50 distinct customers
    "country":   rng.choice(countries, size=N),              # 6 distinct
    "category":  rng.choice(categories, size=N),             # 4 distinct
    "status":    rng.choice(statuses, size=N,
                            p=[0.15, 0.25, 0.55, 0.05]),      # skewed -> RLE-friendly
    "amount":    np.round(rng.gamma(shape=2.0, scale=25.0, size=N), 2),
    "qty":       rng.integers(1, 6, size=N),
})

print(df.shape)
df.head(8)

polars  1.44.1
pyarrow 25.0.0
(500, 7)


order_id,customer,country,category,status,amount,qty
i64,i64,str,str,str,f64,i64
1,1004,"""GB""","""toys""","""shipped""",1.31,3
2,1038,"""US""","""grocery""","""shipped""",6.29,5
3,1032,"""JP""","""grocery""","""shipped""",13.77,4
4,1021,"""JP""","""toys""","""delivered""",40.25,4
5,1021,"""FI""","""books""","""delivered""",56.39,1
6,1042,"""NL""","""toys""","""shipped""",18.83,1
7,1004,"""US""","""toys""","""shipped""",69.89,2
8,1034,"""FI""","""grocery""","""delivered""",62.46,2


### Write it two ways: polars and pyarrow

Both write the *same* logical table. polars' `write_parquet` is the one-liner you'll
reach for day to day; `pyarrow.parquet.write_table` is the lower-level API that
exposes every knob (row-group size, encodings, statistics, versions) — we'll use it
for the internals section.

In [2]:
workdir = tempfile.mkdtemp(prefix="parquet_deepdive_")
print("scratch dir:", workdir)

path_polars = os.path.join(workdir, "orders_polars.parquet")
path_arrow  = os.path.join(workdir, "orders_pyarrow.parquet")

# 1) polars — snappy by default
df.write_parquet(path_polars, compression="snappy")

# 2) pyarrow — go through an Arrow Table. Small row_group_size so a 500-row
#    table splits into several row groups; that makes the internals visible.
table = df.to_arrow()
pq.write_table(table, path_arrow, compression="snappy", row_group_size=150)

for p in (path_polars, path_arrow):
    print(f"{os.path.getsize(p):>7d} bytes   {os.path.basename(p)}")

scratch dir: /tmp/parquet_deepdive_d3lnhlmw
   8843 bytes   orders_polars.parquet
  14091 bytes   orders_pyarrow.parquet


The file exists on disk as a single self-describing blob — data **plus** a footer
of metadata (schema, row-group offsets, per-column statistics). That footer is what
makes Parquet readable without a separate catalog: the file knows its own shape.

In [3]:
# The magic number: a Parquet file begins AND ends with the 4 bytes b"PAR1".
with open(path_arrow, "rb") as fh:
    head = fh.read(4)
    fh.seek(-4, os.SEEK_END)
    tail = fh.read(4)
print("first 4 bytes:", head)
print("last  4 bytes:", tail)
assert head == tail == b"PAR1", "not a Parquet file!"
print("\\nDisk listing:")
for p in sorted(glob.glob(os.path.join(workdir, '*.parquet'))):
    print(f"  {os.path.getsize(p):>7d}  {os.path.basename(p)}")

first 4 bytes: b'PAR1'
last  4 bytes: b'PAR1'
\nDisk listing:
     8843  orders_polars.parquet
    14091  orders_pyarrow.parquet


## 3 · Internals — row groups, column chunks, schema

A Parquet file is a hierarchy:

```
File
 └─ Row Group 1            (a horizontal slice of N rows)
 │   ├─ Column Chunk: order_id   ── Page, Page, ...
 │   ├─ Column Chunk: customer   ── Page, Page, ...
 │   └─ ... one chunk per column
 └─ Row Group 2
 │   └─ ...
 └─ Footer (schema + row-group/column metadata + statistics)  ← read first
```

- **Row group** — a batch of rows (here we forced 150). It's the unit of parallelism
  and the unit of predicate pushdown: a reader can skip an entire row group using its
  statistics.
- **Column chunk** — all values for one column within one row group. Stored
  contiguously → the source of column projection.
- **Page** — the smallest unit inside a chunk; carries the encoding + optional
  compression and its own min/max stats.

`ParquetFile(...).metadata` exposes all of this. Let's read the footer of the
pyarrow file we wrote with `row_group_size=150` (so 500 rows → 4 row groups).

In [4]:
pf = pq.ParquetFile(path_arrow)
meta = pf.metadata

print("created by      :", meta.created_by)
print("format version  :", meta.format_version)
print("num_columns     :", meta.num_columns)
print("num_rows        :", meta.num_rows)
print("num_row_groups  :", meta.num_row_groups)
print()
print("Schema:")
print(pf.schema_arrow)

created by      : parquet-cpp-arrow version 25.0.0
format version  : 2.6
num_columns     : 7
num_rows        : 500
num_row_groups  : 4

Schema:
order_id: int64
customer: int64
country: large_string
category: large_string
status: large_string
amount: double
qty: int64


### Per-column chunk detail

Now the good part. For each row group, walk its column chunks and pull out the
physical facts: encodings actually used, compression codec, compressed vs
uncompressed bytes, and the min/max statistics that power pushdown.

In [5]:
def dump_row_group(meta, rg_index):
    rg = meta.row_group(rg_index)
    print(f"── Row group {rg_index}: {rg.num_rows} rows, "
          f"{rg.total_byte_size} bytes (uncompressed in-memory est.)")
    header = f"  {'column':<10} {'codec':<7} {'comp':>7} {'uncomp':>7} "\
             f"{'ratio':>6}  {'encodings'}"
    print(header)
    for c in range(rg.num_columns):
        col = rg.column(c)
        name = col.path_in_schema
        comp = col.total_compressed_size
        uncomp = col.total_uncompressed_size
        ratio = (uncomp / comp) if comp else float("nan")
        encs = ",".join(str(e) for e in col.encodings)
        print(f"  {name:<10} {col.compression:<7} {comp:>7} {uncomp:>7} "
              f"{ratio:>5.2f}x  {encs}")

dump_row_group(meta, 0)
print()
dump_row_group(meta, 1)

── Row group 0: 150 rows, 4106 bytes (uncompressed in-memory est.)
  column     codec      comp  uncomp  ratio  encodings
  order_id   SNAPPY      863    1444  1.67x  PLAIN,RLE,RLE_DICTIONARY
  customer   SNAPPY      415     589  1.42x  PLAIN,RLE,RLE_DICTIONARY
  country    SNAPPY      152     150  0.99x  PLAIN,RLE,RLE_DICTIONARY
  category   SNAPPY      145     141  0.97x  PLAIN,RLE,RLE_DICTIONARY
  status     SNAPPY      156     152  0.97x  PLAIN,RLE,RLE_DICTIONARY
  amount     SNAPPY     1051    1444  1.37x  PLAIN,RLE,RLE_DICTIONARY
  qty        SNAPPY      181     186  1.03x  PLAIN,RLE,RLE_DICTIONARY

── Row group 1: 150 rows, 4083 bytes (uncompressed in-memory est.)
  column     codec      comp  uncomp  ratio  encodings
  order_id   SNAPPY      862    1444  1.68x  PLAIN,RLE,RLE_DICTIONARY
  customer   SNAPPY      407     573  1.41x  PLAIN,RLE,RLE_DICTIONARY
  country    SNAPPY      152     150  0.99x  PLAIN,RLE,RLE_DICTIONARY
  category   SNAPPY      145     141  0.97x  PLAIN,RLE,

**Reading the encodings column.** pyarrow reports the encodings present in each
chunk. You'll typically see:

- `RLE_DICTIONARY` (a.k.a. dictionary encoding) on the low-cardinality columns
  (`country`, `category`, `status`). Parquet stores a small **dictionary** of the
  distinct values once, then encodes each row as a tiny integer index into it. When
  the same index repeats in a run, **run-length encoding (RLE)** collapses the run to
  a `(value, count)` pair. Six countries across 150 rows → a 6-entry dictionary plus
  a stream of 0–5 indices, RLE-packed. That's where the compression comes from before
  a general-purpose codec even runs.
- `PLAIN` and/or `RLE` on the high-cardinality / near-unique columns (`order_id`,
  `amount`), where a dictionary wouldn't pay for itself.

`RLE` also always encodes the definition/repetition levels that track nulls and
nesting — so you'll see it listed even on columns that are otherwise `PLAIN`.

In [6]:
# Statistics are what make predicate pushdown possible: each column chunk
# carries min/max/null_count, so a reader can PROVE a row group holds no matching
# row without decoding a single value.
print("Per-row-group min/max for 'amount' and 'country':\n")
print(f"  {'rg':>2}  {'rows':>4}  {'amount.min':>10}  {'amount.max':>10}  "
      f"{'country.min':>11}  {'country.max':>11}")
# column order matches the schema: order_id, customer, country, category, status, amount, qty
COL = {name: i for i, name in enumerate(
    [meta.row_group(0).column(i).path_in_schema
     for i in range(meta.num_columns)])}
for rg_i in range(meta.num_row_groups):
    rg = meta.row_group(rg_i)
    amt = rg.column(COL["amount"]).statistics
    cty = rg.column(COL["country"]).statistics
    print(f"  {rg_i:>2}  {rg.num_rows:>4}  {amt.min:>10.2f}  {amt.max:>10.2f}  "
          f"{str(cty.min):>11}  {str(cty.max):>11}")

Per-row-group min/max for 'amount' and 'country':

  rg  rows  amount.min  amount.max  country.min  country.max
   0   150        1.31      169.52           DE           US
   1   150        2.66      214.03           DE           US
   2   150        1.47      218.66           DE           US
   3    50        6.96      204.05           DE           US


### Compression bake-off — same data, four codecs

Encoding (dictionary/RLE) happens *first* and is codec-independent; then a
general-purpose compressor runs over the encoded pages. Parquet supports several.
We write the identical table with each and compare file size. Rules of thumb:

- **uncompressed** — baseline; fastest to write, largest on disk
- **snappy** — the common default: fast, modest ratio. Great for hot query paths
- **zstd** — near-gzip ratios at snappy-like speed; increasingly the preferred default
- **gzip** — high ratio, slower; fine for cold/archival data

In [7]:
rows = []
for codec in ["none", "snappy", "gzip", "zstd"]:
    p = os.path.join(workdir, f"orders_{codec}.parquet")
    t0 = time.perf_counter()
    pq.write_table(table, p, compression=codec, row_group_size=150)
    write_ms = (time.perf_counter() - t0) * 1e3
    size = os.path.getsize(p)
    rows.append({"codec": codec, "bytes": size, "write_ms": round(write_ms, 2)})

bake = pl.DataFrame(rows).sort("bytes")
base = bake.filter(pl.col("codec") == "none")["bytes"][0]
bake = bake.with_columns(
    (pl.col("bytes") / base).round(3).alias("vs_uncompressed"),
    (100 * (1 - pl.col("bytes") / base)).round(1).alias("pct_saved"),
)
bake

codec,bytes,write_ms,vs_uncompressed,pct_saved
str,i64,f64,f64,f64
"""zstd""",12463,0.6,0.698,30.2
"""gzip""",12817,1.31,0.717,28.3
"""snappy""",14091,0.42,0.789,21.1
"""none""",17865,0.62,1.0,0.0


Even on 500 rows the codecs separate cleanly. The savings are modest here because
dictionary+RLE already did most of the work on the low-cardinality columns before the
codec ran — on wide real-world data with long low-cardinality columns, zstd routinely
lands 3–10x over uncompressed. The takeaway: **encoding and compression are two
independent layers**, and Parquet applies both.

## 4 · Why columnar wins — projection and predicate pushdown, measured

Two levers, both enabled by the columnar layout + footer statistics:

1. **Column projection** — ask for 2 columns, read only those 2 column chunks off
   disk. The other columns' bytes are never touched.
2. **Predicate pushdown** — a filter like `amount > 200` is compared against each row
   group's min/max stats; row groups that can't contain a match are **skipped
   entirely**, before any decoding.

To make the effect large and measurable, we write a **wider, taller** table (12
columns × 40k rows, sorted so `amount` clusters into row groups) and let pyarrow
report exactly how many bytes each read pulled off disk.

In [8]:
# Wider/taller table so projection + pushdown have room to show a difference.
M = 40_000
wide = pl.DataFrame({
    "order_id": np.arange(M, dtype=np.int64),
    "country":  rng.choice(countries, size=M),
    "category": rng.choice(categories, size=M),
    "status":   rng.choice(statuses, size=M),
    "customer": rng.integers(1000, 1500, size=M),
    "amount":   np.round(np.sort(rng.gamma(2.0, 50.0, size=M)), 2),  # sorted!
    "qty":      rng.integers(1, 10, size=M),
    "discount": np.round(rng.random(M), 3),
    "tax":      np.round(rng.random(M) * 20, 2),
    "weight_g": rng.integers(50, 5000, size=M),
    "region":   rng.choice(np.array(["north", "south", "east", "west"]), size=M),
    "channel":  rng.choice(np.array(["web", "app", "store"]), size=M),
})
wide_path = os.path.join(workdir, "orders_wide.parquet")
# Many smallish row groups -> pushdown can skip most of them for a selective filter.
pq.write_table(wide.to_arrow(), wide_path, compression="zstd", row_group_size=4000)

wmeta = pq.ParquetFile(wide_path).metadata
print(f"wide file : {os.path.getsize(wide_path):,} bytes")
print(f"columns   : {wmeta.num_columns}")
print(f"rows      : {wmeta.num_rows:,}")
print(f"row groups: {wmeta.num_row_groups}  (~{wmeta.row_group(0).num_rows} rows each)")

wide file : 637,928 bytes
columns   : 12
rows      : 40,000
row groups: 10  (~4000 rows each)


### Lever 1 — column projection

Read **all 12 columns** vs **just 2**, and compare the compressed bytes actually read
(summed from the column-chunk metadata — this is real on-disk I/O, not a timing
guess).

In [9]:
def bytes_for_columns(meta, columns):
    """Sum compressed on-disk bytes for the given columns across all row groups."""
    idx = {meta.row_group(0).column(i).path_in_schema: i
           for i in range(meta.num_columns)}
    total = 0
    for rg_i in range(meta.num_row_groups):
        rg = meta.row_group(rg_i)
        for name in columns:
            total += rg.column(idx[name]).total_compressed_size
    return total

all_cols = [wmeta.row_group(0).column(i).path_in_schema
            for i in range(wmeta.num_columns)]
proj_cols = ["country", "amount"]

full_bytes = bytes_for_columns(wmeta, all_cols)
proj_bytes = bytes_for_columns(wmeta, proj_cols)

print(f"read all {len(all_cols):>2} columns : {full_bytes:>9,} bytes")
print(f"read     {len(proj_cols):>2} columns : {proj_bytes:>9,} bytes")
print(f"projection reads {proj_bytes / full_bytes:.1%} of the data "
      f"({full_bytes / proj_bytes:.1f}x less I/O)")

read all 12 columns :   624,230 bytes
read      2 columns :   106,072 bytes
projection reads 17.0% of the data (5.9x less I/O)


In [10]:
# And it's not just accounting — the reader really only fetches those chunks.
t0 = time.perf_counter()
_ = pq.read_table(wide_path)                       # all columns
t_all = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pq.read_table(wide_path, columns=proj_cols)    # projected
t_proj = time.perf_counter() - t0

print(f"read_table(all)         : {t_all*1e3:7.2f} ms")
print(f"read_table(2 columns)   : {t_proj*1e3:7.2f} ms")

# polars scan_parquet pushes projection down lazily too:
lazy_cols = pl.scan_parquet(wide_path).select(proj_cols).collect()
print("polars projected frame  :", lazy_cols.shape)

read_table(all)         :  119.08 ms
read_table(2 columns)   :    1.61 ms
polars projected frame  : (40000, 2)


### Lever 2 — predicate pushdown (row-group skipping)

`amount` was written **sorted**, so its values cluster: each row group's min/max
covers a narrow, non-overlapping band. A selective filter like `amount > 380` can
therefore be satisfied by touching only the last few row groups — pyarrow uses the
footer statistics to skip the rest.

We prove the skipping happened by counting, from the metadata alone, how many row
groups *could* match, then confirm the filtered read returns those rows.

In [11]:
THRESH = 380.0
amount_idx = {wmeta.row_group(0).column(i).path_in_schema: i
              for i in range(wmeta.num_columns)}["amount"]

candidate_rgs = []
for rg_i in range(wmeta.num_row_groups):
    st = wmeta.row_group(rg_i).column(amount_idx).statistics
    if st.max > THRESH:            # this row group MIGHT contain a match
        candidate_rgs.append(rg_i)

print(f"filter: amount > {THRESH}")
print(f"row groups total       : {wmeta.num_row_groups}")
print(f"row groups that qualify : {len(candidate_rgs)}  -> {candidate_rgs}")
print(f"pushdown skips {1 - len(candidate_rgs)/wmeta.num_row_groups:.0%} of the row groups\n")

# pyarrow filters= applies exactly this pushdown, then a residual row filter.
filtered = pq.read_table(wide_path,
                         columns=["order_id", "country", "amount"],
                         filters=[("amount", ">", THRESH)])
print("rows returned by pushdown read :", filtered.num_rows)

# Cross-check against a full brute-force scan — must agree.
brute = wide.filter(pl.col("amount") > THRESH).height
print("rows via full polars scan      :", brute)
assert filtered.num_rows == brute, "pushdown changed the result set!"
print("pushdown result matches full scan ✓")

filter: amount > 380.0
row groups total       : 10
row groups that qualify : 1  -> [9]
pushdown skips 90% of the row groups



rows returned by pushdown read : 173
rows via full polars scan      : 173
pushdown result matches full scan ✓


In [12]:
# polars expresses the same thing declaratively; its lazy engine pushes BOTH the
# projection and the predicate into the Parquet reader.
q = (pl.scan_parquet(wide_path)
       .filter(pl.col("amount") > THRESH)
       .select(["order_id", "country", "amount"]))
print(q.explain())            # note 'SELECTION' + projected columns pushed to the scan
out = q.collect()
print("\\npolars lazy pushdown rows:", out.height)
out.head()

Parquet SCAN [/tmp/parquet_deepdive_d3lnhlmw/orders_wide.parquet]
PROJECT 3/12 COLUMNS
SELECTION: col("amount") > 380.0
ESTIMATED ROWS: 40000
\npolars lazy pushdown rows: 173


order_id,country,amount
i64,str,f64
39827,"""DE""",380.58
39828,"""JP""",380.63
39829,"""FI""",380.77
39830,"""FI""",381.2
39831,"""NL""",381.27


## 5 · Round-trip — write, read back, verify equality

The whole point of a storage format is that what comes out equals what went in. We
read the original 500-row file back through both engines and assert equality against
the in-memory frame.

In [13]:
# polars round-trip
rt_polars = pl.read_parquet(path_polars)
assert rt_polars.equals(df), "polars round-trip mismatch!"
print("polars  round-trip: equal ✓", rt_polars.shape)

# pyarrow round-trip -> back to polars for comparison.
rt_arrow = pl.from_arrow(pq.read_table(path_arrow))
# pyarrow wrote from the same frame; column order + dtypes should match exactly.
assert rt_arrow.equals(df), "pyarrow round-trip mismatch!"
print("pyarrow round-trip: equal ✓", rt_arrow.shape)

# Schema survived the trip (types are self-describing in the footer):
print("\\nround-tripped dtypes:")
print(rt_polars.schema)

polars  round-trip: equal ✓ (500, 7)
pyarrow round-trip: equal ✓ (500, 7)
\nround-tripped dtypes:
Schema({'order_id': Int64, 'customer': Int64, 'country': String, 'category': String, 'status': String, 'amount': Float64, 'qty': Int64})


### Bonus — SQL over Parquet with DuckDB, no load step

DuckDB reads Parquet files directly as if they were tables — projection and pushdown
included — so you can run SQL over a file on disk without importing it anywhere.

In [14]:
import duckdb
print("duckdb", duckdb.__version__)

res = duckdb.sql(f"""
    SELECT country,
           count(*)              AS orders,
           round(avg(amount), 2) AS avg_amount,
           round(sum(amount), 2) AS total
    FROM read_parquet('{wide_path}')
    WHERE amount > {THRESH}
    GROUP BY country
    ORDER BY total DESC
""").pl()
res

duckdb 1.5.5


country,orders,avg_amount,total
str,i64,f64,f64
"""DE""",49,447.97,21950.42
"""GB""",27,427.36,11538.69
"""FI""",25,438.09,10952.16
"""NL""",24,443.52,10644.44
"""JP""",25,418.95,10473.68
"""US""",23,433.83,9978.1


## 6 · When to use Parquet — and when not to

The file-format series covers four formats; here's where each fits, so you can pick
deliberately.

| Format | Layout | In-memory / on-disk | Sweet spot | Weak spot |
|---|---|---|---|---|
| **Parquet** | Columnar | On-disk | Analytical scans, data-lake storage, cross-engine interchange | Row-level updates, single-record streaming |
| **Arrow** (IPC/Feather) | Columnar | In-memory (also on-disk) | Zero-copy hand-off between processes/languages; ephemeral spill | Long-term storage (larger, less portable than Parquet) |
| **Avro** | Row | On-disk / wire | Streaming append, Kafka messages, schema-evolution-heavy pipelines | Column scans (must read whole rows) |
| **Lance** | Columnar + | On-disk | Vector/ML data, fast random access + versioning, feature stores | Ubiquity — fewer engines read it than Parquet |

**Reach for Parquet when:**

- You're storing tabular data that will be **scanned and aggregated** far more often
  than it's written row-by-row.
- You want the widest possible **engine compatibility** — Spark, Trino, DuckDB,
  polars, pandas, every cloud warehouse.
- You're building a **lake/lakehouse** (Iceberg, Delta, Hudi all sit on Parquet).
- **Size on disk** and **scan I/O** matter — columnar + dictionary/RLE + zstd is hard
  to beat for OLAP.

**Reach for something else when:**

- You need **row-level mutation** or **point lookups** → a database (Postgres),
  or a table format's merge-on-read layer on top of Parquet.
- You're **streaming one record at a time** → Avro (or a log); batch and write Parquet
  downstream.
- You need a **zero-copy in-memory** exchange between steps/languages in one pipeline
  → Arrow. (Parquet on disk ⇄ Arrow in memory is the standard pairing — polars and
  DuckDB use Arrow as their in-memory representation and Parquet as their disk format.)
- You're storing **embeddings / ML features** with heavy random access and versioning
  → Lance.

### The three ideas to carry away

1. **Columnar layout** turns "read 2 of 12 columns" into 2/12 of the I/O — projection.
2. **Footer statistics per row group** let a reader **skip blocks** that can't match a
   filter — predicate pushdown — before decoding anything.
3. **Two compression layers** — type-aware encoding (dictionary/RLE) *then* a
   general-purpose codec (snappy/zstd/gzip) — stack to shrink the file, and you tune
   the codec per workload (snappy hot, zstd/gzip cold).

_Next in the series: `02_format_arrow` — the in-memory columnar sibling that Parquet
loads into._

In [15]:
# Housekeeping: show the scratch dir contents, then leave them for inspection.
# (It's a tempdir; the OS reclaims it. Uncomment to remove now.)
print("Artifacts written under:", workdir)
for p in sorted(glob.glob(os.path.join(workdir, '*.parquet'))):
    print(f"  {os.path.getsize(p):>9,}  {os.path.basename(p)}")
# import shutil; shutil.rmtree(workdir)

Artifacts written under: /tmp/parquet_deepdive_d3lnhlmw
     12,817  orders_gzip.parquet
     17,865  orders_none.parquet
      8,843  orders_polars.parquet
     14,091  orders_pyarrow.parquet
     14,091  orders_snappy.parquet
    637,928  orders_wide.parquet
     12,463  orders_zstd.parquet
